# 02 · Ingeniería de variables y feature store — Sprint 1 (F2)
> **Datos 100 % sintéticos.** AURORA SCORE y Vertex Financial Group son un caso ficticio con fines académicos (SCY1102, Duoc UC). Ejecutar antes `python src/pipeline.py`.

Tarjetas AUR-06 y AUR-08. El caso describe más de 240 variables; el prototipo implementa 48 representativas de las seis fuentes.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
REP = RAIZ / "reports"; FIG = REP / "figures"
leer = lambda n: json.loads((REP / n).read_text(encoding="utf-8"))
from features import CATALOGO

In [2]:
fs = pd.read_parquet(RAIZ / 'data/features/feature_store.parquet')
print(fs.shape)
fs.head()

(42000, 52)


,solicitud_id,pais,fecha_solicitud,cohorte,monto_solicitado_usd,plazo_meses,es_postpago,marca_telefono,gama_equipo,antiguedad_linea_meses,comuna_antena,operador,meses_telecom,recargas_media,recargas_regularidad,...,reclamos_total,devoluciones_total,tasa_reclamos,tasa_devoluciones,compras_tendencia,tiempo_pantallas_seg,velocidad_formulario_seg,n_ediciones_formulario,so_ios,antiguedad_cuenta_red,tamano_red_log,redes_optin,cuota_sobre_recarga,n_fuentes_con_datos,perfil_thin_file
0,1,CL,2025-01-01,historica,117.0,5,0,ZTE Blade,0,39,CL-C27,Entel,6.0,2.333333,0.690161,...,0.0,0.0,0.000000,0.00,-1.000000,393.0,229.0,3,0,59.0,6.006353,1,0.847463,4,0
1,2,MX,2025-01-01,historica,157.0,5,1,Pixel,2,22,MX-C23,Telcel,6.0,2.833333,0.550678,...,1.0,0.0,0.100000,0.00,0.000000,371.0,109.0,0,0,NaN,NaN,0,1.155920,3,0
2,3,CL,2025-01-01,historica,226.0,5,1,Galaxy A,1,65,CL-C19,Movistar,6.0,3.166667,0.534210,...,0.0,0.0,NaN,NaN,NaN,308.0,207.0,5,0,56.0,6.208590,1,1.190876,3,0
3,4,PE,2025-01-01,historica,905.0,6,1,Galaxy A,1,10,PE-C17,Claro,6.0,3.166667,0.681002,...,0.0,1.0,0.000000,0.04,0.333333,381.0,196.0,2,0,72.0,6.371612,1,5.424146,4,0
4,5,CO,2025-01-01,historica,289.0,5,0,Moto G,1,59,CO-C13,Claro,6.0,1.166667,0.460441,...,1.0,0.0,0.111111,0.00,0.333333,466.0,170.0,6,0,NaN,NaN,0,6.468895,3,0


## Catálogo por fuente y versión
`base` = medición sin conciliar · `mitigado` = conciliada o de cobertura · `ambos` = se usa en las dos versiones.

In [3]:
cat = pd.DataFrame(CATALOGO, index=['fuente', 'version', 'descripcion']).T
cat.groupby(['fuente', 'version']).size().unstack(fill_value=0)

version,ambos,base,mitigado
fuente,,,
app,4,0,0
cobertura,0,0,2
marketplace,9,0,0
redes,3,0,0
servicios,4,4,4
solicitud,3,0,0
telecom,15,0,0


## Exclusión digital inversa: la falta de datos no es mal comportamiento

In [4]:
lat = pd.read_parquet(RAIZ / 'data/raw/_latentes_generador.parquet')
res = pd.read_parquet(RAIZ / 'data/raw/resultados.parquet')
d = fs.merge(lat, on='solicitud_id').merge(res, on='solicitud_id')
d.groupby('thin_file').agg(n=('solicitud_id', 'size'), mora_real=('incumplio', 'mean'),
    meses_servicios=('meses_servicios', 'mean'), compras=('compras_total', 'mean'),
    antiguedad_linea=('antiguedad_linea_meses', 'mean')).round(3)

,n,mora_real,meses_servicios,compras,antiguedad_linea
thin_file,,,,,
False,32875,0.110,5.965,10.665,43.674
True,9125,0.107,0.994,0.393,4.496


## Detección de proxies (umbral 0,35)
Correlación de cada variable con los atributos protegidos: razón de correlación (eta) para numéricas y V de Cramér para categóricas.

In [5]:
pd.read_csv(REP / 'correlacion_proxies.csv', index_col=0).head(12)

,genero,zona,tramo_edad,sector_laboral,max,atributo_max,es_proxy
variable,,,,,,,
comuna_antena,0.075,0.421,0.082,0.280,0.421,zona,True
datos_gb_medio,0.056,0.223,0.108,0.284,0.284,sector_laboral,False
monto_servicios_medio,0.039,0.174,0.051,0.257,0.257,sector_laboral,False
ticket_medio,0.031,0.138,0.047,0.223,0.223,sector_laboral,False
pct_canal_corresponsal,0.018,0.206,0.051,0.146,0.206,zona,False
monto_recargas_medio,0.035,0.145,0.073,0.196,0.196,sector_laboral,False
gama_equipo,0.028,0.131,0.042,0.175,0.175,sector_laboral,False
marca_telefono,0.031,0.133,0.031,0.175,0.175,sector_laboral,False
tamano_red_log,0.030,0.132,0.059,0.175,0.175,sector_laboral,False
